# Coronary Segmentation — Colab BUILD notebook (Stage 1)

End-to-end on a **Colab GPU**: get code → install → place data → standardize → nnU-Net **teacher** → cache teacher logits → distill a **TinyU-Net student** → export **ONNX + state_dict**. The student `state_dict` is the portable handoff; CoreML conversion + the clDice gate + on-device benchmark happen **on your Mac** (`docs/COLAB_MAC_SPLIT.md`).

### How to run
1. **Runtime → Change runtime type → GPU** (T4 is enough).
2. Keep `DRY_RUN = True` for the first pass — it runs the *entire* pipeline in a few minutes (5-epoch teacher, 3-epoch student) so you can confirm wiring before spending real GPU time. Set it `False` for a real model.
3. Put **ARCADE** (Zenodo record 10390295, COCO) and **DCA1** under `data/raw/` when the data cell tells you to.
4. **Runtime → Run all**.

All heavy logic lives in the importable `src/*` package — this notebook only orchestrates it.

## 1 · Check the GPU
If this errors or shows no GPU, switch the runtime to GPU first.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > GPU, then Run all.'
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2 · Get the code + install
Clones the repo into the Colab session and installs the **build** env (torch-CUDA + nnU-Net + Ultralytics + ONNX). `coremltools` is **not** installed here — that's the Mac side.

If the repo is **private**, paste a GitHub token (Settings → Developer settings → Personal access tokens, `repo` scope) into `GITHUB_TOKEN`. If it's public, leave it empty.

In [ ]:
import os, sys
GITHUB_TOKEN = ''                                              # '' if repo is PUBLIC; else paste a PAT
REPO_SLUG = 'jugalmodi0111/interventional-imaging-pipeline'
REPO = '/content/interventional-imaging-pipeline'
if not os.path.exists(REPO):
    auth = f'{GITHUB_TOKEN}@' if GITHUB_TOKEN else ''
    !git clone https://{auth}github.com/{REPO_SLUG}.git {REPO}
%cd {REPO}
sys.path.insert(0, REPO)
!pip -q install 'nnunetv2>=2.5' 'monai>=1.3' 'ultralytics>=8.2' SimpleITK \
    onnx onnxscript onnxruntime opencv-python scikit-image scikit-learn pyyaml tqdm
print('\nbuild env ready')

## 3 · Mount Drive + set persistent paths + DRY_RUN
Colab wipes local disk on disconnect, so nnU-Net state, the teacher cache, and `runs/` point at **Google Drive** — a dropped session won't lose the teacher. `DRY_RUN` caps epochs so you can validate the whole path fast.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/intv-img'

DRY_RUN = True            # True = fast wiring check (5-epoch teacher, 3-epoch student). False = real run.

for k, sub in [('nnUNet_raw','nnUNet_raw'), ('nnUNet_preprocessed','nnUNet_preprocessed'),
               ('nnUNet_results','nnUNet_results')]:
    p = f'{DRIVE}/{sub}'; os.makedirs(p, exist_ok=True); os.environ[k] = p
RAW = f'{DRIVE}/data/raw'; os.makedirs(RAW, exist_ok=True)
RUNS = f'{DRIVE}/runs/coronary'; os.makedirs(RUNS, exist_ok=True)
TEACHER_CACHE = f'{DRIVE}/teacher_cache/coronary'; os.makedirs(TEACHER_CACHE, exist_ok=True)
!mkdir -p data && ln -sfn {RAW} data/raw
print('DRY_RUN =', DRY_RUN)
print('raw data   :', RAW)
print('runs       :', RUNS)

## 4 · Place the datasets, then verify
Download and **unzip into Drive** (persists across sessions):
- **ARCADE** (COCO): https://zenodo.org/records/10390295  → unzip to `{RAW}/arcade`
- **DCA1**: http://personal.cimat.mx:8181/~ivan.cruz/DB_Angiograms.html  → unzip to `{RAW}/dca1`

The cell below just checks they're present (it does not download for you). Both are Open access.

In [ ]:
import glob
def nfiles(d): return len(glob.glob(f'{d}/**/*', recursive=True)) if os.path.exists(d) else 0
for name in ['arcade', 'dca1']:
    n = nfiles(f'{RAW}/{name}')
    print(f'{name:8}: {n} files', '' if n else f'  <-- EMPTY: unzip the dataset into {RAW}/{name}')
assert nfiles(f'{RAW}/arcade') and nfiles(f'{RAW}/dca1'), 'Place ARCADE + DCA1 under data/raw first.'

## 5 · Standardize → binary vessel pairs + nnU-Net raw
`arcade_to_coco` unions the 25-region polygons into a binary vessel mask; `dca1_to_nnunet` pairs `X`/`X_gt`. Both write `data/processed/coronary/{img,msk}` (student) **and** the nnU-Net raw dataset (teacher), applying CLAHE.

In [ ]:
!python -m src.data_prep.arcade_to_coco --config configs/coronary_seg.yaml
!python -m src.data_prep.dca1_to_nnunet --config configs/coronary_seg.yaml
print('processed pairs:', len(glob.glob('data/processed/coronary/img/*.png')))
!python -m src.eval.metrics   # smoke-test the metric code (Stage 0 exit)

## 6 · Teacher — nnU-Net v2 (2D)
Sets the accuracy ceiling and doubles as a label-quality oracle. Preprocess, then train fold 0. `DRY_RUN` uses the built-in 5-epoch trainer; a real run uses 250 epochs. `--c` resumes from the last checkpoint on Drive if the session dropped.

*(Upgrade to the ResEnc-M planner later: `-pl nnUNetPlannerResEncM` + train with `-p nnUNetResEncUNetMPlans`.)*

In [ ]:
DATASET_ID = '001'                                            # Dataset001_Coronary from the prep step
TRAINER = 'nnUNetTrainer_5epochs' if DRY_RUN else 'nnUNetTrainer_250epochs'
!nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity
!nnUNetv2_train {DATASET_ID} 2d 0 -tr {TRAINER} --c || nnUNetv2_train {DATASET_ID} 2d 0 -tr {TRAINER}

## 7 · Cache teacher logits
`nnUNetv2_predict --save_probabilities` writes per-case softmax `.npz` — these are the soft targets the student distills toward.

In [ ]:
!nnUNetv2_predict -i $nnUNet_raw/Dataset{DATASET_ID}_Coronary/imagesTr \
                  -o {TEACHER_CACHE} -d {DATASET_ID} -c 2d -f 0 -tr {TRAINER} --save_probabilities
print('teacher npz:', len(glob.glob(f'{TEACHER_CACHE}/*.npz')))

## 8 · Distill → TinyU-Net student (~2-3M params, edge)
`distill()` trains the student against cached teacher logits with sigmoid-KD + hard supervision, and saves a **state_dict** (`student.pt`) — the portable Mac handoff.

In [ ]:
import yaml
from torch.utils.data import DataLoader
from src.models.seg_student import TinyUNet
from src.models.distill import distill, TeacherCacheDataset
from src.eval.metrics import dice, cldice

cfg = yaml.safe_load(open('configs/coronary_seg.yaml'))
ds = TeacherCacheDataset('data/processed/coronary', TEACHER_CACHE, size=cfg['preprocess']['size'])
loader = DataLoader(ds, batch_size=cfg['train']['batch'], shuffle=True, num_workers=2)
student = TinyUNet(in_ch=1, n_classes=1, base=cfg['student']['base_ch'], depth=cfg['student'].get('depth', 4))

def quick_eval(m):
    m.eval()
    with torch.no_grad():
        x, y, _ = ds[0]
        p = torch.sigmoid(m(x[None].cuda())).cpu().numpy().squeeze() >= 0.5
    m.train()
    g = y.numpy().squeeze() > 0.5
    return f'Dice {dice(p, g):.3f} clDice {cldice(p, g):.3f}'

epochs = 3 if DRY_RUN else cfg['train']['epochs']
distill(student, loader, epochs=epochs, lr=cfg['train']['lr'],
        alpha=cfg['distill']['alpha'], T=cfg['distill']['temperature'],
        eval_fn=quick_eval, ckpt=f'{RUNS}/student.pt')

## 9 · Export handoff — ONNX + state_dict
ONNX is for non-Apple / onnxruntime checks; the **state_dict** is what the Mac rebuilds into TinyU-Net before CoreML conversion. Both land on Drive.

In [ ]:
!python -m src.export.to_onnx --weights {RUNS}/student.pt --out {RUNS}/student.onnx
!python -m src.export.quantize_int8 --onnx {RUNS}/student.onnx
!python -m src.eval.edge_benchmark --model {RUNS}/student.onnx
print('\nHANDOFF READY on Drive:')
print('  ', f'{RUNS}/student.pt    (state_dict -> Mac CoreML)')
print('  ', f'{RUNS}/student.onnx  (non-Apple / onnxruntime)')

## 10 · Next — on your Mac (Apple silicon)
Pull `student.pt` from Drive, then:
```bash
make export-coreml   MODEL=runs/coronary/student.pt          # -> student.mlpackage (6-bit palettized)
make validate-coreml CORE=runs/coronary/student.mlpackage WEIGHTS=runs/coronary/student.pt \
                     IMAGES=data/processed/coronary/val/img MASKS=data/processed/coronary/val/msk
make bench-coreml    MODEL=runs/coronary/student.mlpackage
```
`validate-coreml` is the **hard gate**: clDice drop after compression must be ≤ 0.03 (palettization breaks thin vessels even when Dice holds).

When the dry run is green, set **`DRY_RUN = False`** (cell 3) and Run all again for a real model.